# Brain Tumor Segmentation with U-Net
**HackBio Stage Three — Final Submission**

**Dataset:** BraTS2020 — Multi-modal MRI scans with expert-annotated tumor masks
**Model:** U-Net (encoder–decoder with skip connections)
**Framework:** PyTorch

---
### Contents
1. [Task 1 — Data Preparation](#task1)
2. [Task 2 — U-Net Architecture](#task2)
3. [Task 3 — Training](#task3)
4. [Task 4 — Evaluation](#task4)
5. [Task 5 — Visualise Predictions](#task5)
6. [Results Summary](#results)

---
## Running this Notebook

### Google Colab is the default (recommended)

This notebook is designed to run on **Google Colab**, which provides a free NVIDIA GPU via the browser. It can also be run directly from VS Code using the **Google Colab extension**, which connects VS Code to Colab's GPU servers — so you get the familiar VS Code interface without giving up the GPU.

Deep learning models like U-Net require significant computation. A laptop CPU processes one operation at a time, which makes training very slow. A GPU runs thousands of operations simultaneously — the same task that takes hours on a CPU takes minutes on a GPU.

| | Google Colab / Colab Extension (default) | Laptop CPU only |
|---|---|---|
| **Hardware** | Free NVIDIA T4 GPU | CPU only — no GPU |
| **Speed** | ~7 minutes per epoch | ~1 hour per epoch |
| **Data & quality** | 20,000 slices, full resolution, full model | Must cut all of the above to finish in reasonable time |
| **How to run** | Browser (colab.google) or VS Code Colab extension | Run locally via terminal or VS Code |

---
### If you are running on a CPU-only laptop

The notebook defaults are set for Colab/GPU. If you must run locally without a GPU, make these **4 changes** to avoid the notebook taking many hours:

**Change 1 — Limit dataset size** (Task 1, cell `c006`): reduce the cap:
```python
ALL_FILES = ALL_FILES[:3000]   # limits to 3,000 files instead of 20,000
```

**Change 2 — Reduce image resolution** (Task 1, cell `c006`):
```python
IMG_SIZE = 128   # reduce from 240 to save memory
```

**Change 3 — Reduce model capacity** (Task 2, cell `c014`):
```python
C = [32, 64, 128, 256, 512]   # reduce from [64, 128, 256, 512, 1024]
```

**Change 4 — Reduce training epochs** (Task 3, cell `c020`):
```python
NUM_EPOCHS = 5   # reduce from 15
```

> **Note:** These CPU reductions will produce poor segmentation results (large blobs instead of precise tumor outlines). This is expected — the model has not been trained enough on enough data. Run on Colab for meaningful predictions.

## Imports & Setup

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import h5py

print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Hardware:        GPU — {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("Hardware:        CPU")
    print("WARNING: No GPU detected. Training will be very slow.")
    print("On Colab: Runtime → Change runtime type → T4 GPU → Save, then restart.")

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
import os

OUTPUT_DIR = "/content/brain_tumor_segmentation"
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
PLOTS_DIR  = os.path.join(OUTPUT_DIR, "plots")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,  exist_ok=True)

print("Output structure initialised:")
print("  brain_tumor_segmentation/")
print("  ├── models/    ← best_model.pt, checkpoint.pt")
print("  ├── plots/     ← loss_curve.png, prediction_visualisation.png")
print("  └── results_summary.txt")

# Mount Google Drive and mirror the models directory so best_model.pt
# survives a runtime restart (Colab's /content/ is wiped on disconnect).
try:
    from google.colab import drive as _drive
    _drive.mount("/content/drive", force_remount=False)
    DRIVE_MODELS_DIR = "/content/drive/MyDrive/brain_tumor_segmentation/models"
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    print(f"Drive mounted — models will also save to: {DRIVE_MODELS_DIR}")
except Exception:
    DRIVE_MODELS_DIR = None
    print("Drive not available — saving to local runtime only.")

---
<a id="task1"></a>
## Task 1 — Data Preparation

The BraTS2020 dataset provides 4-channel MRI volumes (T1, T1ce, T2, FLAIR) with
3-channel tumour masks per slice.  We use the **FLAIR** channel as input (channel index 3)
and collapse the three mask channels into a single binary tumour mask.

In [ ]:
try:
    import kagglehub
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

# Kaggle credentials are required to download the dataset.
# On Colab:  run kagglehub.login() in a new cell — it opens a browser-based login
# On laptop: place kaggle.json at ~/.kaggle/kaggle.json (Windows: C:\Users\<you>\.kaggle\kaggle.json)
#            Download it from kaggle.com → Settings → API → Create New Token

path = kagglehub.dataset_download("awsaf49/brats2020-training-data")
print("Dataset path:", path)

In [ ]:
ARCHIVE_DIR = os.path.join(path, "BraTS2020_training_data", "content", "data") + os.sep
ALL_FILES   = [f for f in os.listdir(ARCHIVE_DIR) if f.endswith(".h5")]
print(f"Total .h5 files found: {len(ALL_FILES)}")

ALL_FILES = ALL_FILES[:15000]
print(f"Using: {len(ALL_FILES)} files for this run")

IMG_SIZE = 240

In [ ]:
class BraTSDataset(Dataset):
    def __init__(self, file_list, archive_dir, augment=False):
        self.archive_dir = archive_dir
        self.augment     = augment
        self.files = []

        for fname in file_list:
            fpath = os.path.join(archive_dir, fname)
            with h5py.File(fpath, "r") as f:
                img_shape  = f["image"].shape
                mask_shape = f["mask"].shape

            if img_shape[:2] != (240, 240) or mask_shape[:2] != (240, 240):
                continue
            if img_shape[2] != 4 or mask_shape[2] != 3:
                continue

            self.files.append(fpath)

        print(f"Valid slices retained: {len(self.files)}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        with h5py.File(self.files[idx], "r") as f:
            img  = f["image"][:]
            mask = f["mask"][:]

        flair = torch.tensor(img[:, :, 3], dtype=torch.float32).unsqueeze(0)
        flair = (flair - flair.min()) / (flair.max() - flair.min() + 1e-8)

        tumour_mask = mask.sum(axis=2).clip(0, 1)
        tumour_mask = torch.tensor(tumour_mask, dtype=torch.float32).unsqueeze(0)

        flair       = F.interpolate(flair.unsqueeze(0),       size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False).squeeze(0)
        tumour_mask = F.interpolate(tumour_mask.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE), mode="nearest").squeeze(0)

        if self.augment:
            if random.random() > 0.5:
                flair       = torch.flip(flair,       dims=[2])
                tumour_mask = torch.flip(tumour_mask, dims=[2])
            if random.random() > 0.5:
                flair       = torch.flip(flair,       dims=[1])
                tumour_mask = torch.flip(tumour_mask, dims=[1])

        return {"image": flair, "mask": tumour_mask}

In [ ]:
# Split by volume ID to prevent patient-level data leakage.
# BraTS2020 filenames are volume_X_slice_Y.h5 — group all slices for the
# same volume together before splitting so no patient appears in both sets.
volume_ids = sorted(set(f.split("_slice_")[0] for f in ALL_FILES))
random.shuffle(volume_ids)
split      = int(0.90 * len(volume_ids))
train_vols = set(volume_ids[:split])
val_vols   = set(volume_ids[split:])

train_files = [f for f in ALL_FILES if f.split("_slice_")[0] in train_vols]
val_files   = [f for f in ALL_FILES if f.split("_slice_")[0] in val_vols]

print(f"Volumes — train: {len(train_vols)}  val: {len(val_vols)}")
print(f"Slices  — train: {len(train_files)}  val: {len(val_files)}")

In [ ]:
train_dataset = BraTSDataset(train_files, ARCHIVE_DIR, augment=True)
val_dataset   = BraTSDataset(val_files,   ARCHIVE_DIR, augment=False)

In [ ]:
BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

if len(train_dataset) == 0:
    raise RuntimeError("Training dataset is empty — check that the .h5 files downloaded correctly and pass the shape/mask filters in BraTSDataset.")

batch = next(iter(train_loader))
print("Image batch shape:", batch["image"].shape)
print("Mask batch shape: ", batch["mask"].shape)

### Verify Dataset — Display 5 Image–Mask Pairs

In [ ]:
def show_image_mask_pairs(dataset, n=5, title="Sample Image–Mask Pairs"):
    n = min(n, len(dataset))
    indices = random.sample(range(len(dataset)), n)
    fig, axes = plt.subplots(n, 2, figsize=(6, 3 * n))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    for row, idx in enumerate(indices):
        sample = dataset[idx]
        image  = sample["image"].squeeze().numpy()
        mask   = sample["mask"].squeeze().numpy()

        axes[row, 0].imshow(image, cmap="gray")
        axes[row, 0].set_title(f"FLAIR  (sample {idx})", fontsize=9)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask, cmap="hot")
        axes[row, 1].set_title(f"Tumour Mask  (sample {idx})", fontsize=9)
        axes[row, 1].axis("off")

    plt.tight_layout()
    plt.show()

show_image_mask_pairs(train_dataset, n=5, title="Task 1 — 5 Training Image–Mask Pairs")

---
<a id="task2"></a>
## Task 2 — U-Net Architecture

The U-Net consists of:
- **Encoder** — four successive double-conv + max-pool blocks that halve spatial resolution
- **Bottleneck** — the deepest double-conv block (1024 channels)
- **Decoder** — four transpose-conv upsampling blocks that restore spatial resolution
- **Skip connections** — feature maps from each encoder level are concatenated to the corresponding decoder level
- **Output layer** — 1×1 convolution projecting to a single-channel logit map

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv     = DoubleConv(in_ch, out_ch)

    def forward(self, x_dec, x_enc):
        x_dec = self.upsample(x_dec)
        dy = x_enc.size(2) - x_dec.size(2)
        dx = x_enc.size(3) - x_dec.size(3)
        x_dec = F.pad(x_dec, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        return self.conv(torch.cat([x_enc, x_dec], dim=1))


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        # CPU only: reduce to [32, 64, 128, 256, 512] to lower memory usage on a local machine
        C = [64, 128, 256, 512, 1024]

        self.enc0       = DoubleConv(in_channels, C[0])
        self.enc1       = EncoderBlock(C[0], C[1])
        self.enc2       = EncoderBlock(C[1], C[2])
        self.enc3       = EncoderBlock(C[2], C[3])
        self.bottleneck = EncoderBlock(C[3], C[4])
        self.dec1       = DecoderBlock(C[4], C[3])
        self.dec2       = DecoderBlock(C[3], C[2])
        self.dec3       = DecoderBlock(C[2], C[1])
        self.dec4       = DecoderBlock(C[1], C[0])
        self.head       = nn.Conv2d(C[0], out_channels, kernel_size=1)

    def forward(self, x):
        s0 = self.enc0(x)          # IMG_SIZE    × C[0]
        s1 = self.enc1(s0)         # IMG_SIZE/2  × C[1]
        s2 = self.enc2(s1)         # IMG_SIZE/4  × C[2]
        s3 = self.enc3(s2)         # IMG_SIZE/8  × C[3]
        b  = self.bottleneck(s3)   # IMG_SIZE/16 × C[4]
        x  = self.dec1(b,  s3)
        x  = self.dec2(x,  s2)
        x  = self.dec3(x,  s1)
        x  = self.dec4(x,  s0)
        return self.head(x)        # IMG_SIZE    × out_channels

In [ ]:
model = UNet(in_channels=1, out_channels=1).to(device)
print(model)

In [ ]:
with torch.no_grad():
    test_input  = torch.randn(1, 1, IMG_SIZE, IMG_SIZE).to(device)
    test_output = model(test_input)

print("Input shape: ", test_input.shape)
print("Output shape:", test_output.shape)
assert test_output.shape == test_input.shape, "Output shape must match mask shape!"
print("Shape check passed.")

---
<a id="task3"></a>
## Task 3 — Training

**Loss:** Combined Dice + Binary Cross-Entropy (50/50 weight) — directly optimises the overlap metric
**Optimizer:** Adam (lr = 1e-3) with `ReduceLROnPlateau` scheduler (halves LR after 2 epochs without improvement)
**Gradient clipping:** max norm 1.0 (stabilises early training on large masks)
**Early stopping:** patience = 3 epochs

In [ ]:
class DiceBCELoss(nn.Module):
    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets)
        prob  = torch.sigmoid(logits)
        inter = (prob * targets).sum(dim=(1, 2, 3))
        union = prob.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice  = 1 - ((2 * inter + 1) / (union + 1 + 1e-6)).mean()
        return 0.5 * bce + 0.5 * dice

criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:
    import subprocess; subprocess.run(["pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, n_batches = 0.0, 0
    pbar = tqdm(loader, desc="  Train", leave=False, unit="batch")
    for batch in pbar:
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix(loss=f"{total_loss / n_batches:.4f}")

    return total_loss / n_batches


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, n_batches = 0.0, 0
    for batch in tqdm(loader, desc="  Val  ", leave=False, unit="batch"):
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device)
        logits = model(images)
        total_loss += criterion(logits, masks).item()
        n_batches  += 1
    return total_loss / n_batches

In [ ]:
NUM_EPOCHS = 10
PATIENCE   = 3

history = {"train_loss": [], "val_loss": []}
best_val_loss     = float("inf")
epochs_no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    t_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_loss = validate(model, val_loader, criterion, device)

    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)

    scheduler.step(v_loss)

    if v_loss < best_val_loss - 1e-4:
        best_val_loss     = v_loss
        epochs_no_improve = 0
        local_best = os.path.join(MODELS_DIR, "best_model.pt")
        torch.save(model.state_dict(), local_best)
        if DRIVE_MODELS_DIR:
            import shutil
            shutil.copy(local_best, os.path.join(DRIVE_MODELS_DIR, "best_model.pt"))
        tag = " ✓ best"
    else:
        epochs_no_improve += 1
        tag = f" (no improvement {epochs_no_improve}/{PATIENCE})"

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": v_loss,
    }, os.path.join(MODELS_DIR, "checkpoint.pt"))

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train loss: {t_loss:.4f} | Val loss: {v_loss:.4f} | LR: {lr:.2e}{tag}")

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} — val loss did not improve for {PATIENCE} epochs.")
        break

best_model_path = os.path.join(MODELS_DIR, "best_model.pt")
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, weights_only=True))
    print("Best model weights restored.")
else:
    print("WARNING: best_model.pt not found — using current weights.")

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, history["train_loss"], label="Train loss", linewidth=2)
plt.plot(epochs, history["val_loss"],   label="Val loss",   linewidth=2, linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Dice+BCE Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
<a id="task4"></a>
## Task 4 — Evaluation

Evaluate on the **validation set** using:
- **Dice Score** — measures overlap between prediction and ground truth
- **IoU (Jaccard Index)** — intersection over union

Both metrics range from 0 (no overlap) to 1 (perfect overlap).

In [ ]:
def dice_score(logits, targets, threshold=0.5, eps=1e-7):
    probs  = torch.sigmoid(logits)
    preds  = (probs > threshold).float()
    inter  = (preds * targets).sum(dim=(1, 2, 3))
    union  = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return ((2 * inter + eps) / (union + eps)).mean().item()


def iou_score(logits, targets, threshold=0.5, eps=1e-7):
    probs  = torch.sigmoid(logits)
    preds  = (probs > threshold).float()
    inter  = (preds * targets).sum(dim=(1, 2, 3))
    union  = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - inter
    return ((inter + eps) / (union + eps)).mean().item()

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    dice_scores, iou_scores = [], []
    for batch in loader:
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device)
        logits = model(images)
        dice_scores.append(dice_score(logits, masks, threshold))
        iou_scores.append(iou_score(logits, masks, threshold))
    return np.mean(dice_scores), np.mean(iou_scores)


val_dice, val_iou = evaluate(model, val_loader, device)
print(f"Validation Dice Score : {val_dice:.4f}")
print(f"Validation IoU Score  : {val_iou:.4f}")

---
<a id="task5"></a>
## Task 5 — Visualise Predictions

For each of the 5 selected validation images, display:

| Column 1 | Column 2 | Column 3 |
|---|---|---|
| Original FLAIR MRI | Ground Truth Mask | Predicted Mask |

In [ ]:
@torch.no_grad()
def visualise_predictions(model, dataset, device, n=5, threshold=0.5, save_path=None):
    model.eval()
    n = min(n, len(dataset))
    indices = random.sample(range(len(dataset)), n)

    fig, axes = plt.subplots(n, 3, figsize=(10, 3.5 * n))
    fig.suptitle("Task 5 — Prediction Visualisation\n"
                 "Left: FLAIR MRI | Centre: Ground Truth | Right: Predicted Mask",
                 fontsize=12, fontweight="bold")

    col_titles = ["Original MRI (FLAIR)", "Ground Truth Mask", "Predicted Mask"]
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=10, fontweight="bold")

    for row, idx in enumerate(indices):
        sample = dataset[idx]
        image  = sample["image"].unsqueeze(0).to(device)
        mask   = sample["mask"].squeeze().numpy()

        logit = model(image)
        pred  = (torch.sigmoid(logit) > threshold).float().squeeze().cpu().numpy()

        img_np = sample["image"].squeeze().numpy()

        axes[row, 0].imshow(img_np, cmap="gray")
        axes[row, 0].set_ylabel(f"Sample {idx}", fontsize=8)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask, cmap="hot", vmin=0, vmax=1)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(img_np, cmap="gray")
        masked_pred = np.ma.masked_where(pred == 0, pred)
        axes[row, 2].imshow(masked_pred, cmap="autumn", alpha=0.6, vmin=0, vmax=1)
        axes[row, 2].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()

visualise_predictions(model, val_dataset, device, n=5,
                      save_path=os.path.join(PLOTS_DIR, "prediction_visualisation.png"))

---
<a id="results"></a>
## Results Summary

In [ ]:
if not history["val_loss"]:
    print("No epochs completed — skipping summary.")
else:
    final_train_loss = history["train_loss"][-1]
    final_val_loss   = history["val_loss"][-1]
    best_val_loss    = min(history["val_loss"])
    best_epoch       = history["val_loss"].index(best_val_loss) + 1
    epochs_trained   = len(history["train_loss"])

    print("=" * 50)
    print("         FINAL RESULTS SUMMARY")
    print("=" * 50)
    print(f"  Epochs trained       : {epochs_trained}/{NUM_EPOCHS}")
    print(f"  Final train loss     : {final_train_loss:.4f}")
    print(f"  Final val loss       : {final_val_loss:.4f}")
    print(f"  Best val loss        : {best_val_loss:.4f}  (epoch {best_epoch})")
    print("-" * 50)
    print(f"  Validation Dice Score: {val_dice:.4f}")
    print(f"  Validation IoU Score : {val_iou:.4f}")
    print("=" * 50)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
epochs_range = range(1, len(history["train_loss"]) + 1)
ax.plot(epochs_range, history["train_loss"], label="Train loss", linewidth=2)
ax.plot(epochs_range, history["val_loss"],   label="Val loss",   linewidth=2, linestyle="--")
ax.axvline(best_epoch, color="gray", linestyle=":", linewidth=1.5, label=f"Best val epoch ({best_epoch})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Dice+BCE Loss")
ax.set_title("Training and Validation Loss — Results Summary")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "loss_curve.png"), dpi=150, bbox_inches="tight")
print(f"Saved: {os.path.join(PLOTS_DIR, 'loss_curve.png')}")
plt.show()

In [ ]:
import zipfile

summary_text = f"""
============================================================
              BRAIN TUMOUR SEGMENTATION — RESULTS SUMMARY
============================================================
  Dataset            : BraTS2020 (FLAIR channel, binary mask)
  Model              : U-Net

  Epochs trained     : {epochs_trained}/{NUM_EPOCHS}
  Final train loss   : {final_train_loss:.4f}
  Final val loss     : {final_val_loss:.4f}
  Best val loss      : {best_val_loss:.4f}  (epoch {best_epoch})
------------------------------------------------------------
  Validation Dice Score : {val_dice:.4f}
  Validation IoU Score  : {val_iou:.4f}
============================================================
"""

with open(os.path.join(OUTPUT_DIR, "results_summary.txt"), "w") as f:
    f.write(summary_text)

print(summary_text)

# Print output tree with file sizes
print("Output tree:")
for root, dirs, files in sorted([(r, d, f) for r, d, f in os.walk(OUTPUT_DIR)]):
    level  = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "    " * level
    label  = os.path.basename(root) + "/" if level > 0 else "brain_tumor_segmentation/"
    print(f"{indent}{label}")
    for i, fname in enumerate(sorted(files)):
        connector = "└── " if i == len(files) - 1 else "├── "
        fsize     = os.path.getsize(os.path.join(root, fname))
        size_str  = f"{fsize / 1e6:.1f} MB" if fsize >= 1e6 else f"{fsize / 1e3:.0f} KB"
        print(f"{indent}    {connector}{fname}  ({size_str})")

# Zip and download as a single file
zip_path = "/content/brain_tumor_segmentation.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for fname in files:
            fpath   = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, "/content")
            zf.write(fpath, arcname)

zip_size = os.path.getsize(zip_path)
print(f"\nZipped → brain_tumor_segmentation.zip  ({zip_size / 1e6:.1f} MB)")

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
    print("Download triggered — check your browser downloads.")
except ImportError:
    print(f"Running locally — files already at: {OUTPUT_DIR}")

### Model Architecture Summary

| Component | Details |
|---|---|
| Input | 1-channel FLAIR MRI, 240×240 |
| Encoder | 4 × (MaxPool2d + DoubleConv): 64 → 128 → 256 → 512 |
| Bottleneck | MaxPool2d + DoubleConv → 1024 channels |
| Decoder | 4 × (ConvTranspose2d + skip concat + DoubleConv): 512 → 256 → 128 → 64 |
| Output | 1×1 Conv → single-channel logit map |
| Loss | Dice + BCE (50/50) |
| Optimiser | Adam (lr = 1e-3), ReduceLROnPlateau (factor=0.5, patience=2) |
| Augmentation | Random horizontal + vertical flips (train only) |
| Early stopping | Patience = 3 epochs |
| Metric threshold | 0.5 |